# NEURAL NETWORKS

This notebook covers the neural network algorithms from chapter 18 of the book *Artificial Intelligence: A Modern Approach*, by Stuart Russel and Peter Norvig. 

## NEURAL NETWORK ALGORITHM

### Overview

Although the Perceptron may seem like a good way to make classifications, it is a linear classifier (which, roughly, means it can only draw straight lines to divide spaces) and therefore it can be stumped by more complex problems. To solve this issue we can extend Perceptron by employing multiple layers of its functionality. The construct we are left with is called a Neural Network, or a Multi-Layer Perceptron, and it is a non-linear classifier. It achieves that by combining the results of linear functions on each layer of the network.

Similar to the Perceptron, this network also has an input and output layer; however, it can also have a number of hidden layers. These hidden layers are responsible for the non-linearity of the network. The layers are comprised of nodes. Each node in a layer (excluding the input one), holds some values, called *weights*, and takes as input the output values of the previous layer. The node then calculates the dot product of its inputs and its weights and then activates it with an *activation function* (e.g. sigmoid activation function). Its output is then fed to the nodes of the next layer. Note that sometimes the output layer does not use an activation function, or uses a different one from the rest of the network. The process of passing the outputs down the layer is called *feed-forward*.

After the input values are fed-forward into the network, the resulting output can be used for classification. The problem at hand now is how to train the network (i.e. adjust the weights in the nodes). To accomplish that we utilize the *Backpropagation* algorithm. In short, it does the opposite of what we were doing up to this point. Instead of feeding the input forward, it will track the error backwards. So, after we make a classification, we check whether it is correct or not, and how far off we were. We then take this error and propagate it backwards in the network, adjusting the weights of the nodes accordingly. We will run the algorithm on the given input/dataset for a fixed amount of time, or until we are satisfied with the results. The number of times we will iterate over the dataset is called *epochs*. In a later section we take a detailed look at how this algorithm works.

NOTE: Sometimes we add another node to the input of each layer, called *bias*. This is a constant value that will be fed to the next layer, usually set to 1. The bias generally helps us "shift" the computed function to the left or right.

![neural_net](images/neural_net.png)

## PERCEPTRON CLASSIFIER

### Overview

The Perceptron is a linear classifier. It works the same way as a neural network with no hidden layers (just input and output). First it trains its weights given a dataset and then it can classify a new item by running it through the network.

Its input layer consists of the the item features, while the output layer consists of nodes (also called neurons). Each node in the output layer has *n* synapses (for every item feature), each with its own weight. Then, the nodes find the dot product of the item features and the synapse weights. These values then pass through an activation function (usually a sigmoid). Finally, we pick the largest of the values and we return its index.

Note that in classification problems each node represents a class. The final classification is the class/node with the max output value.

Below you can see a single node/neuron in the outer layer. With *f* we denote the item features, with *w* the synapse weights, then inside the node we have the dot product and the activation function, *g*.

![perceptron](images/perceptron.png)

We will start by implementing perceptron learner on the Iris dataset. Let's first import the Iris dataset by using the scikit-learn library with the following code:

In [ ]:

from sklearn import datasets

iris_dataset = datasets.load_iris()
from sklearn.model_selection import train_test_split
# set random seed to 1, so that we can reproduce the results
iris_dataset_train_X, iris_dataset_test_X, iris_dataset_train_y, iris_dataset_test_y = train_test_split(iris_dataset.data, iris_dataset.target, test_size=0.2, random_state=1)

print(iris_dataset_train_X.shape)
print(iris_dataset_train_y.shape)
print(iris_dataset_test_X.shape)
print(iris_dataset_test_y.shape)


### Implementation

The implementation of `Linear perceptron classifier` in scikit-learn is done by using the `Perceptron` class. By default, the `Perceptron` model does not require a learning rate, not regularized(penalized), and updates its model only on mistake.

We train the Perceptron model on the dataset, and we classify a new item with the following measurements: [7.5, 2.5, 6.5, 2.0].

In [ ]:
from sklearn.linear_model import Perceptron
perceptron_clf = Perceptron()
perceptron_clf.fit(iris_dataset_train_X, iris_dataset_train_y)

example = [[7.5, 2.5, 6.5, 2.0]]
predict_result = perceptron_clf.predict(example)
print(iris_dataset.target_names[predict_result[0]])

# Image Classification

In this tutorial, we will use Sklearn to classify images, using the MNIST dataset. Besides the algorithms we have seen in the previous tutorials, we will also give a try to the multi-layer perceptron (MLP) algorithm.

In [3]:
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import sklearn

# MNIST dataset
The MNIST database (Modified National Institute of Standards and Technology database) is a large collection of handwritten digits. It has a training set of 60,000 examples, and a test set of 10,000 examples. The images were centered in a 28x28 image by computing the center of mass of the pixels, and translating the image so as to position this point at the center of the 28x28 field. The resulting images contain grey levels as a result of the anti-aliasing technique used by the normalization algorithm.

We will load the datasets by directly reading the images from the disk. Let's first give a look at the directory structure:
<pre>
└─MNIST
    ├─testing
    │  ├─0
    │  ├─1
    │  ├─2
    │  ├─3
    │  ├─4
    │  ├─5
    │  ├─6
    │  ├─7
    │  ├─8
    │  └─9
    └─training
        ├─0
        ├─1
        ├─2
        ├─3
        ├─4
        ├─5
        ├─6
        ├─7
        ├─8
        └─9
<pre>

By looking at the directory structure, we could figure out that the images are stored in directories named after the class they belong to. We will use this information to load the images and their corresponding labels.

# Loading the dataset

In this section, we will use `os.listdir` to list all files in the specified directory, and `PIL.Image` to read the images. We will also use `numpy` to store the images and their corresponding labels.

## `os` module

First, let's list the directories in the training and testing directories:

In [ ]:
train_dir = 'MNIST/training'
test_dir = 'MNIST/testing'

print('folders name in training directory:', os.listdir(train_dir))
print('folders name in testing directory:', os.listdir(test_dir))

Thus, we get the names of all the subdirectories in the training and testing directories. Before we proceed, let's have a look at some basic usage of `os` library. The following code snippet lists all the files in the `train_dir` directory:

In [ ]:
for label in os.listdir(train_dir): # loop through each folder in the training directory. The folder name is the label of the images in that folder
    sub_folder_path = os.path.join(train_dir, label) # os.path.join is used to join one or more path segments
    images = os.listdir(sub_folder_path) # this will return a list of all the images in the sub folder
    print(label, 'folder contains', len(images), 'images')

## `PIL` and `numpy` modules
We use the `PIL` (pillow) module to read the images and the `numpy` module to store the images and their corresponding labels.  Let's start by reading a image from the disk:


In [ ]:
image_path = os.path.join(train_dir, '0', '1.png')
print('image path:', image_path)

img = Image.open(image_path) # open the image file
img = np.array(img) # convert the image to a numpy array
plt.imshow(img, cmap='gray') # display the image in grayscale

To create the datasets which could be used by the classifiers, we usually flatten the images and store them in a 2D array: each row of the array represents an image, and each column represents a pixel. First, we could transform the image to a 1D array by using the `numpy` module:

In [ ]:
# for grayscale images, the shape will be (height, width)
img_vec = img.reshape(-1) # flatten the image to a vector
print('image shape:', img.shape)
print('image vector shape:', img_vec.shape)

Please note that, since images in MNIST dataset are grayscale, the images are stored in a 2D array. For images with 3 channels (RGB), the process is similar, but the images are stored in a 3D array.

In [ ]:
# for color images, the shape will be (height, width, 3)
img_rgb = Image.open('test.png')
img_rgb = np.array(img_rgb)
plt.imshow(img_rgb)

img_rgb_vec = img_rgb.reshape(-1)
print('image shape:', img_rgb.shape)
print('image vector shape:', img_rgb_vec.shape)

## Helper function
By using the above code snippets, let's create a helper function to load the images and their corresponding labels:

In [9]:
import random
def load_images(folder_path, sample_ratio=0.33):
    '''
    Load a random subset of images from the given folder_path.
    Returns flattened images (vectors) and labels.
    :param folder_path: Path to the dataset
    :param sample_ratio: Fraction of images to load
    '''
    vectors = []
    labels = []
    
    # Iterate through each class sub-folder
    for label in os.listdir(folder_path):
        sub_folder_path = os.path.join(folder_path, label)
        
        # Skip if not a directory
        if not os.path.isdir(sub_folder_path):
            continue
            
        # Get all image filenames in the sub-folder
        all_images = os.listdir(sub_folder_path)
        
        # Calculate the number of images to sample from this class
        sample_size = int(len(all_images) * sample_ratio)
        
        # Randomly select a subset of image paths
        sampled_images = random.sample(all_images, sample_size)
        
        for image_path in sampled_images:
            # Open the image file
            image = Image.open(os.path.join(sub_folder_path, image_path))
            # Convert the image to a numpy array
            image = np.array(image)
            # Flatten the image to a vector
            vector = image.reshape(-1)
            
            vectors.append(vector)
            labels.append(label)
            
    return np.array(vectors), np.array(labels)

It might take a while to load all the images:

In [ ]:
training_vectors, training_labels = load_images(train_dir)
testing_vectors, testing_labels = load_images(test_dir)

print('training image_vectors shape:', training_vectors.shape)
print('training labels shape:', training_labels.shape)
print('testing image_vectors shape:', testing_vectors.shape)
print('testing labels shape:', testing_labels.shape)


# Multilayer Perceptron (MLP)
Multi-layer Perceptron (MLP) is a supervised learning algorithm that training on a dataset. Given a set of features and targets, it can learn a non-linear function approximator for either classification or regression. It is different from logistic regression, in that between the input and the output layer, there can be one or more non-linear layers, called hidden layers. The following figure shows a simple MLP with one hidden layer:

![MLP](images/MLP.jpg) 



We will use the `MLPClassifier` class from the `sklearn.neural_network` module to classify the images. To create a MLP classifier, we need to specify the number of hidden layers, the number of neurons in each hidden layer, the activation function, and the learning rate. The following code snippet shows how to create a MLP classifier:

In [ ]:
from sklearn.neural_network import MLPClassifier
import time

# create a multi-layer perceptron classifier with 1 input layer, 2 hidden layer with 20 neurons, and 1 output layer
# use tanh activation function and 1e-3 as the learning rate
# max_iter is the maximum number of iterations to train the model, you can increase this value if the model is not converging
# verbose=True will print the progress of the training
mlp_clf = MLPClassifier(hidden_layer_sizes=(20, 20), 
                        activation='tanh', 
                        learning_rate_init=1e-3, 
                        max_iter=20, 
                        random_state=1, 
                        verbose=True)

start_time = time.time()
mlp_clf.fit(training_vectors, training_labels)

# predict the labels of the testing images
predicted_labels = mlp_clf.predict(testing_vectors)
print('predicted labels:', predicted_labels)
print(f"run time: {time.time() - start_time:.4f} seconds")

# calculate the accuracy of the model
accuracy = np.mean(predicted_labels == testing_labels)
print('accuracy:', accuracy)

The activation function could be chosen from `identity`, `logistic`, `tanh`, and `relu`. More information about the MLP classifier could be found in the [official documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html).

## EXERCISES
### 1. Adjust the parameters of the MLP classifier and observe the changes in the accuracy. Try to reach accuracy higher than 95% in 60s.
You may notice that the accuracy of the MLP classifier is not very high. You can try to adjust the parameters of the MLP classifier to improve the accuracy. For example, you can try to increase the number of hidden layers, the number of neurons in each hidden layer, the activation function, and the learning rate. You can also try to increase the number of maximum iterations to allow the classifier to converge. Please notice that MLP classifier use CPU to train, so it may take a long time to train the model.

In [12]:
# todo

### 2. Try to use other classifiers which we have seen in the previous tutorials to classify the images, such as `Decision Tree`. Follow the same steps as we did for the MLP classifier.

In [ ]:
from sklearn import tree

decision_tree_clf = tree.DecisionTreeClassifier(criterion='entropy')
decision_tree_clf = decision_tree_clf.fit(training_vectors, training_labels)

predicted_labels = decision_tree_clf.predict(testing_vectors)
accuracy = np.mean(predicted_labels == testing_labels)
print('Decision Tree accuracy:', accuracy)

In [ ]:
"""linear regression"""

In [ ]:
"""kNN classifier"""

In [ ]:
"""logistic regression"""


In [ ]:
"""naive bayes"""
